# EDA: Amazon + Steam reviews

Этот notebook — тонкий слой над production-кодом. Сбор и построение EDA выполняет `DataCollectionAgent`; здесь читаются сохранённые таблицы и графики. Live-сбор выключен по умолчанию, чтобы случайный `Run All` не обращался к сети.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'config.yaml').exists() and (candidate / 'agents').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents.common import load_frame
from agents.data_collection_agent import DataCollectionAgent

collector = DataCollectionAgent(ROOT / 'config.yaml')
print(f'Project root: {ROOT}')

## Воспроизводимый сбор

Чтобы явно запустить два online-источника из notebook, переключите флаг. Для обычной работы предпочтительнее `python run_pipeline.py --review-mode required --force`: CLI дополнительно сохраняет состояние пайплайна.

In [ ]:
RUN_LIVE_COLLECTION = False

if RUN_LIVE_COLLECTION:
    collected = collector.run()  # production implementation; requires network
    print(f'Collected {len(collected)} rows from {collected.source.nunique()} sources')
else:
    print('Live collection is disabled. Existing artifacts will be inspected.')

In [ ]:
candidates = [
    ROOT / 'data/raw/reviews_raw.parquet',
    ROOT / 'data/raw/reviews_raw.csv',
    ROOT / 'data/fixtures/reviews_fixture.csv',
]
dataset_path = next((path for path in candidates if path.exists()), None)

if dataset_path is None:
    reviews = None
    print('Нет raw-артефакта или fixture. Сначала запустите collection stage.')
else:
    reviews = load_frame(dataset_path)
    mode = 'SMOKE FIXTURE' if 'fixtures' in dataset_path.parts else 'COLLECTED DATA'
    print(f'{mode}: {dataset_path} · rows={len(reviews)}')
    display(reviews.head())
    display(pd.DataFrame({'dtype': reviews.dtypes.astype(str), 'missing': reviews.isna().sum()}))

## Артефакты EDA

Production agent сохраняет распределение классов и источников, длины текстов, top-20 слов и PNG-графики в `reports/eda`. Если каталог ещё не создан, ячейка ниже завершится без ошибки.

In [ ]:
eda_dir = ROOT / 'reports/eda'
tables = [
    'class_distribution.csv',
    'source_distribution.csv',
    'text_lengths.csv',
    'top_20_words.csv',
]
available = False
for name in tables:
    path = eda_dir / name
    if path.exists():
        available = True
        print(f'\n{name}')
        display(pd.read_csv(path).head(20))

for name in ['class_distribution.png', 'source_distribution.png', 'text_length_distribution.png', 'top_20_words.png']:
    path = eda_dir / name
    if path.exists():
        available = True
        display(Image(filename=str(path)))

if not available:
    print('EDA-артефакты пока отсутствуют. Запустите live collection через CLI или RUN_LIVE_COLLECTION=True.')

## Что интерпретировать после запуска

1. Проверить, что присутствуют оба независимых источника и оба класса.
2. Сравнить длины Amazon и Steam-текстов: сильный сдвиг влияет на очистку и качество TF-IDF.
3. Сопоставить top-20 слов с доменами; доменные токены могут позволить модели угадывать источник вместо тональности.
4. В отчёт переносить только фактические числа из артефактов, а fixture явно маркировать как smoke data.